In [ ]:
import dataclasses
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import mne
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from scripts.notebook_helpers import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    analyzers_to_datasets,
    compute_wavelet_datasets,
    load_analyzers,
)
from mne.viz import plot_topomap  # noqa: E402
from src.analysis.isc import (  # noqa: E402
    compute_loo_isc,
    compute_sliding_window_isc,
    compute_sliding_window_isc_spearman,
)
from src.analysis.mean_variance import (  # noqa: E402
    compute_intersubject_stats,
    compute_windowed_stats,
)
from src.visualization.isc_plots import (  # noqa: E402
    plot_multiscale_sliding_window_isc,
    plot_band_multiscale_sliding_window_isc,
)
from src.visualization.mean_variance_plots import (  # noqa: E402
    plot_timeseries,
    plot_variance_distribution,
    plot_windowed_analysis,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Wavelet Power Exploration

Exploratory analysis of **wavelet-transformed EEG data** (power representation).

This notebook loads pre-computed (or freshly computed) wavelet transforms in 4-D
format `(n_subjects, n_channels, n_frequencies, n_times)` and runs the following
sketch analyses:

1. Grand-average spectral profile
2. Time–frequency map (spectrogram)
3. Per-band power time course
4. Intersubject variance in the wavelet domain
5. Wavelet-domain LOO-ISC (per band)

See `README.md` in this directory for the rationale behind each step and ideas
for future extensions.

## Configuration

In [ ]:
# ── Dataset switch ───────────────────────────────────────────────────────────
# Choose which experiment to analyse:
#   ExperimentNames.PSILO_MUSIC → psilocybin music-listening (CLASSIC / PSYTRANCE)
#   ExperimentNames.ASSR        → auditory steady-state response (no music dimension)
EXPERIMENT_NAME = ExperimentNames.ASSR

if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration

# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers.
WAVELET_FREQ_RESOLUTION_HZ = (WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / (
    WAVELET_N_FREQS - 1
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
# Always limit to the first N individuals (fast interactive exploration).
# Set to None to use all subjects.
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
# Slice the EEG data before wavelet computation.  The small subset is stored
# in the wavelet cache so subsequent runs simply reload from disk without any
# re-computation.  Set to None to disable the respective slicing.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples (from ~91 000)

# ── Sliding-window settings ──────────────────────────────────────────────────
WINDOW_FINE_SEC: float = 1.0  # fine window for multi-scale ISC (seconds)
WINDOW_MED_SEC: float = 5.0  # medium window for multi-scale ISC (seconds)
WINDOW_LARGE_SEC: float = 15.0  # large window for multi-scale ISC (seconds)
STEP_SEC: float = 2.5  # step for mean-variance windowed analysis (seconds)

# ── Scope ─────────────────────────────────────────────────────────────────────
RUN_BROADBAND = True
RUN_PER_BAND = True

# ── Storage directory (notebook-local cache, avoids large processed-data writes) ────
# Namespaced by experiment so ASSR and music wavelet caches don't collide.
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "03-wavelet-analysis"
    / "wavelet_cache"
    / EXPERIMENT_NAME.value
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOT_PERCENTILE_CLIP: int = 99  # clip upper percentile in mean/variance plots
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "03-wavelet-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "power"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment             : {EXPERIMENT_NAME.value}")
print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Reshape to 4D          : {RESHAPE_FREQUENCY_DIM}")
print(
    "Subject subset         : "
    + (
        f"{N_SUBJECTS_SUBSET} individuals"
        if N_SUBJECTS_SUBSET is not None
        else "all individuals"
    )
)
print(
    "Channel subset         : "
    + (
        f"{N_CHANNELS_SUBSET} channels"
        if N_CHANNELS_SUBSET is not None
        else "all channels"
    )
)
print(
    "Time subset            : "
    + (f"{N_TIMES_SUBSET} samples" if N_TIMES_SUBSET is not None else "all samples")
)

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, {ad.n_samples} samples"
    )

## Load or compute wavelet transforms

### Broadband

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets: dict = {}

if RUN_BROADBAND:
    broadband_datasets = compute_wavelet_datasets(
        datasets=datasets,
        analyzers=analyzers,
        experiment_name=EXPERIMENT_NAME,
        freqs=FREQS,
        representation=REPRESENTATION,
        keep_frequency_dim=KEEP_FREQUENCY_DIM,
        reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
        wavelet_dir=WAVELET_DIR / "broadband",
        reuse_wavelets=REUSE_WAVELETS,
    )
    for label, ad in broadband_datasets.items():
        source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
        print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

### Per-band

Each EEG band is stored in `WAVELET_DIR/band_<name>/`.

In [ ]:
band_datasets: dict[str, dict] = {}  # band_name -> {label: AnalysisData}

if RUN_PER_BAND:
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
        n_freqs = max(
            2,
            int(round((h_freq - l_freq) / WAVELET_FREQ_RESOLUTION_HZ)) + 1,
        )
        band_freqs = np.linspace(l_freq, h_freq, n_freqs)
        band_datasets[band] = compute_wavelet_datasets(
            datasets=datasets,
            analyzers=analyzers,
            experiment_name=EXPERIMENT_NAME,
            freqs=band_freqs,
            representation=REPRESENTATION,
            keep_frequency_dim=KEEP_FREQUENCY_DIM,
            reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
            wavelet_dir=WAVELET_DIR / f"band_{band}",
            reuse_wavelets=REUSE_WAVELETS,
        )
        for label, ad in band_datasets[band].items():
            source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
            print(f"[{band:6s}] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `bb_data`
(broadband, 4-D) and the derived scalars.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]
# LABEL = list(broadband_datasets.keys())[1]  # uncomment for second music type

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## 1 — Grand-Average Spectral Profile

Average wavelet power across subjects and time → `(n_channels, n_freqs)`;
then average over channels → `(n_freqs,)` spectrum.

Provides a sanity check: classical EEG peaks (alpha ~10 Hz) should be visible.

In [ ]:
# Grand average: mean over subjects & time → (n_channels, n_freqs)
grand_avg_ch_freq = bb_data.mean(axis=(0, 3))  # (n_channels, n_freqs)
# Channel average → (n_freqs,)
grand_avg_spectrum = grand_avg_ch_freq.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: spectrum
axes[0].plot(FREQS, grand_avg_spectrum, color="steelblue", linewidth=1.5)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Mean wavelet power")
axes[0].set_title(f"Grand-average spectrum — {LABEL}")
# Shade canonical bands
band_colors = {
    "delta": "#d4e6f1",
    "theta": "#d5f5e3",
    "alpha": "#fdebd0",
    "beta": "#fadbd8",
    "gamma": "#e8daef",
}
for band, (lo, hi) in FREQUENCY_BANDS.items():
    axes[0].axvspan(lo, hi, alpha=0.25, color=band_colors.get(band, "grey"), label=band)
axes[0].legend(fontsize=8, loc="upper right")

# Panel B: channel × frequency image
im = axes[1].imshow(
    grand_avg_ch_freq,
    aspect="auto",
    origin="lower",
    extent=[FREQS[0], FREQS[-1], 0, n_channels],
    cmap="viridis",
)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Channel index")
axes[1].set_title(f"Channel × frequency power — {LABEL}")
fig.colorbar(im, ax=axes[1], label="Power")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "spectral_profile.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2 — Time–Frequency Map (Spectrogram)

Channel-averaged and subject-averaged wavelet power as a 2-D image
`(n_freqs × n_times)`.  Reveals how spectral content evolves over the stimulus.

In [ ]:
# Average over subjects and channels → (n_freqs, n_times)
tfr_map = bb_data.mean(axis=(0, 1))  # (n_freqs, n_times)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    tfr_map,
    aspect="auto",
    origin="lower",
    extent=[time[0], time[-1], FREQS[0], FREQS[-1]],
    cmap="inferno",
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"Time–frequency map (mean wavelet power) — {LABEL}")
fig.colorbar(im, ax=ax, label="Power")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "time_frequency_map.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3 — Per-Band Power Time Course

For each band, average wavelet power over the band's frequency bins, then compute
the channel-averaged group mean ± SD across subjects.  Directly comparable to the
broadband time series in notebook `01-*` but now frequency-resolved.

In [ ]:
fig, axes = plt.subplots(
    len(FREQUENCY_BANDS), 1, figsize=(14, 3 * len(FREQUENCY_BANDS)), sharex=True
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, (band, (lo, hi)) in zip(axes, FREQUENCY_BANDS.items()):
    # Select frequency indices within this band
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    # Average over band freqs → (n_subjects, n_channels, n_times)
    band_power = bb_data[:, :, band_mask, :].mean(axis=2)
    # Z-score across time per subject per channel (same as 01-* analyses)
    bp_mean = band_power.mean(axis=2, keepdims=True)
    bp_std = band_power.std(axis=2, keepdims=True)
    band_power_z = (band_power - bp_mean) / (bp_std + 1e-10)
    # Channel average → (n_subjects, n_times)
    band_ch_avg = band_power_z.mean(axis=1)
    group_mean = band_ch_avg.mean(axis=0)
    group_std = band_ch_avg.std(axis=0)

    # Clip to high percentile to remove outliers
    upper = np.percentile(group_mean + group_std, PLOT_PERCENTILE_CLIP)
    lower = np.percentile(group_mean - group_std, 100 - PLOT_PERCENTILE_CLIP)

    ax.plot(time, group_mean, color="steelblue", linewidth=0.8, label="Mean")
    ax.fill_between(
        time,
        group_mean - group_std,
        group_mean + group_std,
        alpha=0.25,
        color="steelblue",
        label="±1 SD",
    )
    ax.set_ylim(lower, upper)
    ax.set_ylabel("Z-scored power")
    ax.set_title(f"{band} ({lo:.0f}–{hi:.0f} Hz)")
    ax.legend(fontsize=7, loc="upper right")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Per-band wavelet power time course (z-scored) — {LABEL}", fontsize=13, y=1.01
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "band_power_timecourse.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4 — Intersubject Mean-Variance Analysis (Wavelet Domain)

For each frequency band, collapse the frequency dimension to get
`(n_subjects, n_channels, n_times)`, then run the same mean-variance
analysis as in `01-raw-mean-variance-analysis`: timeseries, variance
distribution, and windowed synchrony detection.

The broadband version uses the mean across **all** wavelet frequencies.


In [ ]:
# ── Broadband mean-variance ──────────────────────────────────────────────────
# Average over all frequencies → (n_subjects, n_channels, n_times)
bb_data_3d = bb_data.mean(axis=2)
# Z-score per subject × channel along time to normalise amplitude scale
# differences before computing intersubject variance (matches the z-scoring
# used in plot_intersubject_variance).
bb_mean = bb_data_3d.mean(axis=2, keepdims=True)
bb_std = bb_data_3d.std(axis=2, keepdims=True)
bb_data_z = (bb_data_3d - bb_mean) / (bb_std + 1e-10)
bb_stats = compute_intersubject_stats(bb_data_z)

bb_mv_dir = PLOTS_DIR / "broadband" / "mean_variance"
bb_mv_dir.mkdir(parents=True, exist_ok=True)

plot_timeseries(
    bb_stats,
    sfreq,
    LABEL,
    save_path=bb_mv_dir / f"timeseries_{LABEL}.png" if SAVE_PLOTS else None,
)
plot_variance_distribution(
    bb_stats["inter_var"],
    LABEL,
    save_path=bb_mv_dir / f"variance_distribution_{LABEL}.png" if SAVE_PLOTS else None,
)
df_wins_bb = compute_windowed_stats(
    bb_stats,
    n_times=bb_data_3d.shape[2],
    sfreq=sfreq,
    window_sec=WINDOW_MED_SEC,
    step_sec=STEP_SEC,
)
plot_windowed_analysis(
    bb_stats,
    df_wins_bb,
    sfreq,
    LABEL,
    WINDOW_MED_SEC,
    10.0,
    step_sec=STEP_SEC,
    save_path_bar=bb_mv_dir / f"windowed_bar_{LABEL}.png" if SAVE_PLOTS else None,
    save_path_overlay=bb_mv_dir / f"windowed_overlay_{LABEL}.png"
    if SAVE_PLOTS
    else None,
)

### 4.1 — Per-Band Mean-Variance

For each frequency band, apply the same mean-variance analysis on the
band-averaged wavelet power `(n_subjects, n_channels, n_times)`.


In [ ]:
# ── Per-band mean-variance ────────────────────────────────────────────────────
band_mv_dir = PLOTS_DIR / "bands" / "mean_variance"
band_mv_dir.mkdir(parents=True, exist_ok=True)

for band, (lo, hi) in FREQUENCY_BANDS.items():
    if (
        not RUN_PER_BAND
        or band not in band_datasets
        or LABEL not in band_datasets[band]
    ):
        continue
    ad_band = band_datasets[band][LABEL]
    # Collapse frequency dimension → (n_subjects, n_channels, n_times)
    band_3d = ad_band.data.mean(axis=2) if ad_band.data.ndim == 4 else ad_band.data
    # Z-score per subject × channel along time (matches plot_intersubject_variance).
    band_mean = band_3d.mean(axis=2, keepdims=True)
    band_std = band_3d.std(axis=2, keepdims=True)
    band_data_z = (band_3d - band_mean) / (band_std + 1e-10)
    band_stats = compute_intersubject_stats(band_data_z)
    band_label = f"{LABEL} / {band}"

    plot_timeseries(
        band_stats,
        ad_band.sfreq,
        band_label,
        save_path=band_mv_dir / f"{band}_timeseries_{LABEL}.png"
        if SAVE_PLOTS
        else None,
    )
    plot_variance_distribution(
        band_stats["inter_var"],
        band_label,
        save_path=band_mv_dir / f"{band}_variance_distribution_{LABEL}.png"
        if SAVE_PLOTS
        else None,
    )
    df_wins_band = compute_windowed_stats(
        band_stats,
        n_times=band_3d.shape[2],
        sfreq=ad_band.sfreq,
        window_sec=WINDOW_MED_SEC,
        step_sec=STEP_SEC,
    )
    plot_windowed_analysis(
        band_stats,
        df_wins_band,
        ad_band.sfreq,
        band_label,
        WINDOW_MED_SEC,
        10.0,
        step_sec=STEP_SEC,
        save_path_bar=band_mv_dir / f"{band}_windowed_bar_{LABEL}.png"
        if SAVE_PLOTS
        else None,
        save_path_overlay=band_mv_dir / f"{band}_windowed_overlay_{LABEL}.png"
        if SAVE_PLOTS
        else None,
    )

---
## 5 — Wavelet-Domain LOO-ISC (Per Band)

For each band, compute LOO-ISC on band-averaged power
`(n_subjects, n_channels, n_times)` — direct counterpart of `02-*` ISC but in
the wavelet domain.  We expect stimulus-tracking bands (e.g. delta/theta for
rhythm) to show higher ISC.

In [ ]:
band_mean_iscs: dict[str, np.ndarray] = {}

for band, (lo, hi) in FREQUENCY_BANDS.items():
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    band_power_3d = bb_data[:, :, band_mask, :].mean(axis=2)
    _, mean_loo_isc = compute_loo_isc(band_power_3d)
    band_mean_iscs[band] = mean_loo_isc  # (n_channels,)
    print(
        f"  {band:6s}  mean LOO-ISC = {mean_loo_isc.mean():.4f}"
        f"  (median = {np.median(mean_loo_isc):.4f})"
    )

In [ ]:
# Bar chart of per-band mean ISC
bands_list = list(band_mean_iscs.keys())
means = [band_mean_iscs[b].mean() for b in bands_list]
medians = [np.median(band_mean_iscs[b]) for b in bands_list]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bands_list))
width = 0.35
ax.bar(x - width / 2, means, width, label="Mean", color="steelblue")
ax.bar(x + width / 2, medians, width, label="Median", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(bands_list)
ax.set_ylabel("LOO-ISC (channel average)")
ax.set_title(f"Per-band wavelet LOO-ISC — {LABEL}")
ax.legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "wavelet_loo_isc_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.1 — LOO-ISC distribution per band

Histogram of per-channel mean LOO-ISC for each band, clipped at the 99th
percentile to suppress outlier channels.

In [ ]:
fig, axes = plt.subplots(
    1, len(FREQUENCY_BANDS), figsize=(3.5 * len(FREQUENCY_BANDS), 4), sharey=True
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, band in zip(axes, FREQUENCY_BANDS):
    vals = band_mean_iscs[band]
    clip = np.percentile(vals, 99)
    ax.hist(
        vals[vals <= clip], bins=30, color="steelblue", edgecolor="white", alpha=0.8
    )
    ax.axvline(
        vals.mean(),
        color="red",
        linestyle="--",
        linewidth=1,
        label=f"mean={vals.mean():.3f}",
    )
    ax.set_xlabel("LOO-ISC")
    ax.set_title(band)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Number of channels")
fig.suptitle(f"Wavelet LOO-ISC distributions — {LABEL}", fontsize=13, y=1.02)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "wavelet_loo_isc_distributions.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## 6 — Topographic Mapping

Project per-channel mean LOO-ISC for each frequency band onto the scalp montage.
This reveals which brain regions show the strongest frequency-specific synchrony.

Requires MNE channel position info from the loaded data.

In [ ]:
# Get channel info from the analyzer's MNE Info object
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
# Subset to match the actual number of channels used (e.g. after N_CHANNELS_SUBSET)
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

bands_list_topo = list(FREQUENCY_BANDS.keys())
fig, axes = plt.subplots(
    1, len(bands_list_topo), figsize=(3.5 * len(bands_list_topo), 4)
)
for ax, band in zip(axes, bands_list_topo):
    if band not in band_mean_iscs:
        ax.set_visible(False)
        continue
    isc_values = band_mean_iscs[band]  # (n_channels,)
    # plot_topomap requires values to match info channels
    im, _ = plot_topomap(isc_values, info, axes=ax, show=False, cmap="RdBu_r")
    ax.set_title(band, fontsize=10)

fig.suptitle(f"Topographic LOO-ISC per band — {LABEL}", fontsize=12)
plt.colorbar(im, ax=axes[-1], label="mean LOO-ISC")
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / f"topomap_isc_{LABEL}.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7 — Time–Frequency ISC

Compute LOO-ISC at every `(frequency, time)` cell by flattening the frequency dimension
into features and applying per-feature LOO-ISC.

Result is a 2-D ISC spectrogram: `(n_freqs, n_times)`.

In [ ]:
# Broadband data: (n_subjects, n_channels, n_freqs, n_times)
# Reshape to (n_subjects, n_channels * n_freqs, n_times) for LOO-ISC
n_subjects, n_channels, n_freqs_bb, n_times = bb_data.shape
bb_flat = bb_data.reshape(n_subjects, n_channels * n_freqs_bb, n_times)

# Compute per-(channel, freq) ISC -> (n_channels * n_freqs,)
_, tf_isc_flat = compute_loo_isc(bb_flat)

# Reshape back to (n_channels, n_freqs) then average over channels
tf_isc_ch_freq = tf_isc_flat.reshape(n_channels, n_freqs_bb)
tf_isc_freq = tf_isc_ch_freq.mean(axis=0)  # (n_freqs,)

# Also compute per-timepoint ISC via sliding approach (channel-averaged)
# Use band-averaged broadband for a TF-ISC map: compute per-band ISC per time window
# Here we do a per-time ISC heatmap using precomputed per-freq ISC (static)
FREQS_BB = FREQS  # reuse the FREQS variable defined in the config cell

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(FREQS_BB, tf_isc_freq, color="steelblue", lw=2)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Mean LOO-ISC (channel-averaged)")
ax.set_title(f"LOO-ISC vs. Frequency — {LABEL}")
sns.despine()
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"tf_isc_freq_profile_{LABEL}.png", dpi=150, bbox_inches="tight"
    )
plt.show()

---
## 8 — Multi-Scale Sliding-Window Wavelet ISC

Computes LOO-ISC in three window sizes (fine / medium / large) on the
frequency-averaged wavelet power `(n_subjects, n_channels, n_times)`.
Produces a stair-step overlay of all three scales plus a per-channel
ISC heatmap (matching the approach from `02-isc-broadband-analysis`).


In [ ]:
# ── Broadband multi-scale sliding-window ISC ─────────────────────────────────
# Frequency-averaged 3D power (already computed above as bb_data_3d)

bb_sw_dir = PLOTS_DIR / "broadband" / "sliding_window"
bb_sw_dir.mkdir(parents=True, exist_ok=True)

isc_fine, times_fine = compute_sliding_window_isc(
    bb_data_3d, window_sec=WINDOW_FINE_SEC, step_sec=WINDOW_FINE_SEC / 2, sfreq=sfreq
)
isc_med, times_med = compute_sliding_window_isc(
    bb_data_3d, window_sec=WINDOW_MED_SEC, step_sec=WINDOW_MED_SEC / 2, sfreq=sfreq
)
isc_large, times_large = compute_sliding_window_isc(
    bb_data_3d, window_sec=WINDOW_LARGE_SEC, step_sec=WINDOW_LARGE_SEC / 2, sfreq=sfreq
)
isc_spear_med, _ = compute_sliding_window_isc_spearman(
    bb_data_3d, window_sec=WINDOW_MED_SEC, step_sec=WINDOW_MED_SEC / 2, sfreq=sfreq
)

plot_multiscale_sliding_window_isc(
    LABEL,
    isc_fine,
    times_fine,
    isc_med,
    times_med,
    isc_large,
    times_large,
    isc_spear_med,
    sfreq,
    n_times,
    window_fine_sec=WINDOW_FINE_SEC,
    window_med_sec=WINDOW_MED_SEC,
    window_large_sec=WINDOW_LARGE_SEC,
    save_path_bar=bb_sw_dir / f"sw_isc_bar_{LABEL}.png" if SAVE_PLOTS else None,
    save_path_overlay=bb_sw_dir / f"sw_isc_overlay_{LABEL}.png" if SAVE_PLOTS else None,
    save_path_comparison=bb_sw_dir / f"sw_isc_pearson_vs_spearman_{LABEL}.png"
    if SAVE_PLOTS
    else None,
)

### 8.1 — Per-Band Multi-Scale Sliding-Window ISC

For each frequency band, compute multi-scale sliding-window ISC on the
band-averaged wavelet power and produce the 3-scale overlay + heatmap.


In [ ]:
# ── Per-band multi-scale sliding-window ISC ───────────────────────────────────
band_sw_dir = PLOTS_DIR / "bands" / "sliding_window"
band_sw_dir.mkdir(parents=True, exist_ok=True)

band_sw_fine: dict = {}
band_sw_med: dict = {}
band_sw_large: dict = {}
band_sw_spear: dict = {}

for band, (lo, hi) in FREQUENCY_BANDS.items():
    if (
        not RUN_PER_BAND
        or band not in band_datasets
        or LABEL not in band_datasets[band]
    ):
        continue
    ad_band = band_datasets[band][LABEL]
    band_3d = ad_band.data.mean(axis=2) if ad_band.data.ndim == 4 else ad_band.data
    tc_fine, t_fine = compute_sliding_window_isc(
        band_3d,
        window_sec=WINDOW_FINE_SEC,
        step_sec=WINDOW_FINE_SEC / 2,
        sfreq=ad_band.sfreq,
    )
    tc_med, t_med = compute_sliding_window_isc(
        band_3d,
        window_sec=WINDOW_MED_SEC,
        step_sec=WINDOW_MED_SEC / 2,
        sfreq=ad_band.sfreq,
    )
    tc_large, t_large = compute_sliding_window_isc(
        band_3d,
        window_sec=WINDOW_LARGE_SEC,
        step_sec=WINDOW_LARGE_SEC / 2,
        sfreq=ad_band.sfreq,
    )
    tc_sp, _ = compute_sliding_window_isc_spearman(
        band_3d,
        window_sec=WINDOW_MED_SEC,
        step_sec=WINDOW_MED_SEC / 2,
        sfreq=ad_band.sfreq,
    )
    band_sw_fine[band] = (tc_fine, t_fine)
    band_sw_med[band] = (tc_med, t_med)
    band_sw_large[band] = (tc_large, t_large)
    band_sw_spear[band] = (tc_sp, t_med)

if band_sw_fine:
    plot_band_multiscale_sliding_window_isc(
        LABEL,
        band_sw_fine,
        band_sw_med,
        band_sw_large,
        band_sw_spear,
        sfreq=sfreq,
        n_times=n_times,
        window_fine_sec=WINDOW_FINE_SEC,
        window_med_sec=WINDOW_MED_SEC,
        window_large_sec=WINDOW_LARGE_SEC,
        save_path_dir=band_sw_dir if SAVE_PLOTS else None,
    )

---
## 9 — Cross-Frequency Coupling

Explore whether power in one band is correlated with the power in another band.
For each pair of bands, compute the Pearson correlation between their channel-averaged,
subject-mean power time courses.

A high correlation between delta and alpha power, for example, could indicate
nested oscillatory dynamics.

In [ ]:
band_names = list(FREQUENCY_BANDS.keys())
n_bands = len(band_names)

# Build per-band mean power time course: (band, n_times)
band_power_tc: dict[str, np.ndarray] = {}
for band in band_names:
    if band not in band_datasets or LABEL not in band_datasets[band]:
        continue
    ad = band_datasets[band][LABEL]
    # Mean over subjects, channels, freqs -> (n_times,)
    band_power_tc[band] = ad.data.mean(axis=(0, 1, 2))

valid_bands = [b for b in band_names if b in band_power_tc]
n_valid = len(valid_bands)

# Compute pairwise correlation matrix
cfc_matrix = np.full((n_valid, n_valid), np.nan)
for i, b1 in enumerate(valid_bands):
    for j, b2 in enumerate(valid_bands):
        cfc_matrix[i, j] = np.corrcoef(band_power_tc[b1], band_power_tc[b2])[0, 1]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cfc_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(n_valid))
ax.set_yticks(range(n_valid))
ax.set_xticklabels(valid_bands, rotation=45)
ax.set_yticklabels(valid_bands)
ax.set_title(f"Cross-Band Power Correlation — {LABEL}")
plt.colorbar(im, ax=ax, label="Pearson r")
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / f"cross_freq_coupling_{LABEL}.png", dpi=150, bbox_inches="tight"
    )
plt.show()

---
## 10 — Power–Phase Joint Analysis

Overlay per-band power ISC and phase ISC to identify bands where both amplitude
and timing are synchronised vs. bands where only one modality is synchronised.

Phase ISC is approximated here as ISC computed on `cos(phase)` of the band-averaged
wavelet phase (loaded from the phase wavelet cache).

In [ ]:
# Load per-band phase data (wavelet_phase representation)
phase_band_iscs: dict[str, np.ndarray] = {}
phase_wavelet_dir = WAVELET_DIR  # same top-level directory

# Phase ISC requires per-band wavelet caches; skip when only broadband
# data was computed (i.e. RUN_PER_BAND=False in the config cell).
if RUN_PER_BAND:
    for band, (lo, hi) in FREQUENCY_BANDS.items():
        band_dir = phase_wavelet_dir / f"band_{band}"
        if not band_dir.exists():
            print(f"Phase wavelet cache not found for band {band!r}, skipping.")
            continue
        phase_band_datasets_b = compute_wavelet_datasets(
            datasets=datasets,
            analyzers=analyzers,
            experiment_name=EXPERIMENT_NAME,
            freqs=np.linspace(
                lo,
                hi,
                max(2, int(round((hi - lo) / WAVELET_FREQ_RESOLUTION_HZ)) + 1),
            ),
            representation="phase",
            keep_frequency_dim=KEEP_FREQUENCY_DIM,
            reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
            reuse_wavelets=True,
            wavelet_dir=band_dir,
        )
        if LABEL in phase_band_datasets_b:
            ph_ad = phase_band_datasets_b[LABEL]
            # cos(phase) projection then LOO-ISC
            cos_phase = np.cos(
                ph_ad.data.mean(axis=2)
            )  # mean over freqs -> (n_subj, n_ch, n_times)
            _, phase_band_iscs[band] = compute_loo_isc(cos_phase)

# Comparison bar chart
if phase_band_iscs:
    common_bands = [
        b for b in band_names if b in band_mean_iscs and b in phase_band_iscs
    ]
    power_means = [band_mean_iscs[b].mean() for b in common_bands]
    phase_means = [phase_band_iscs[b].mean() for b in common_bands]
    x = np.arange(len(common_bands))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, power_means, width, label="Power ISC", color="steelblue")
    ax.bar(x + width / 2, phase_means, width, label="Phase ISC (cos φ)", color="tomato")
    ax.set_xticks(x)
    ax.set_xticklabels(common_bands)
    ax.set_ylabel("Mean LOO-ISC")
    ax.set_title(f"Power vs. Phase ISC per Band — {LABEL}")
    ax.legend()
    sns.despine()
    plt.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"power_phase_joint_{LABEL}.png", dpi=150, bbox_inches="tight"
        )
    plt.show()
else:
    print("No phase wavelet cache found. Run the phase wavelet pipeline first.")

---
## Summary

The arrays are available for further analysis:

- `broadband_datasets[LABEL].data` — 4-D broadband wavelet power
- `band_datasets[band][LABEL].data` — 4-D per-band wavelet power
- `band_mean_iscs[band]` — per-channel LOO-ISC for each band

Analyses implemented in this notebook:

1. Grand-average spectral profile
2. Time–frequency map (spectrogram)
3. Per-band power time course
4. Intersubject variance in the wavelet domain
5. Wavelet-domain LOO-ISC per band
6. Topographic mapping of per-band LOO-ISC
7. Time–frequency ISC (LOO-ISC vs. frequency profile)
8. Sliding-window wavelet ISC per band
9. Cross-frequency coupling (band-power correlation matrix)
10. Power–phase joint analysis (power ISC vs. phase ISC per band)

See `README.md` in this directory for ideas on extending these analyses
(Placebo vs. Psilocybin condition comparison, permutation-based statistical testing, etc.).